In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd().parent
MASTER = BASE_DIR / "data" / "master"
df_original = pd.read_csv(MASTER / "abuja_rental_master_V2.csv")

df = df_original.copy()

In [3]:
print(df.shape)
df.head()

(4300, 8)


,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At,Source
0,Kubwa,1,600000.0,Apartment,Self Contain,"Studio Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji
1,Kubwa,1,1500000.0,Apartment,Apartment,"1bdrm Block of Flats in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji
2,Life Camp,1,1200000.0,Apartment,Self Contain,"Studio Apartment in Efab City Estate, Life Cam...",2026-05-30 13:25:47,Jiji
3,Kubwa,1,1200000.0,Apartment,Apartment,"1bdrm Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji
4,Life Camp,2,1140000.0,Apartment,Apartment,Furnished 2bdrm Apartment in Life Camp for rent,2026-05-30 13:25:47,Jiji


## Feature Audit & Strategy


### Current Columns:
1. `District` (Categorical) -> **Encode.** group into tiers (Luxury, Mid, Affordable) to give the model a broader understanding, and then One-Hot Encode the specific districts.
2. `Bedrooms` (Numerical) -> **Retain.** Already clean.
3. `Price (Per Annum)` (Numerical/Target) -> **Transform.** Because it is heavily right-skewed, we create a `Log_Price` column.
4. `Property Category` (Categorical) -> **Encode.** Broad category (Apartment, Duplex, Bungalow). We One-Hot Encode this.
5. `Property Type` (Categorical) -> **Drop.** overlaps heavily with Property Category. 
6. `Location` (Text) -> **Drop.**  
7. `Scraped_At` (Date) -> **Drop.** 
8. `Source` (Text) -> **Drop.** 

### District Tiers

In [4]:
# Calculate the median price per district
district_medians = df.groupby('District')['Price (Per Annum)'].median()

def assign_tier(median_price):
    if median_price >= 15_000_000:
        return 'Tier 1 - Luxury'
    elif median_price >= 5_000_000:
        return 'Tier 2 - Mid-Market'
    else:
        return 'Tier 3 - Affordable'

# Map the tiers back to df
tier_mapping = district_medians.apply(assign_tier).to_dict()
df['District_Tier'] = df['District'].map(tier_mapping)

print(df['District_Tier'].value_counts())

District_Tier
Tier 2 - Mid-Market    2087
Tier 1 - Luxury        1696
Tier 3 - Affordable     517
Name: count, dtype: int64


### Apply log transformation to the price (per annum)

In [5]:
df['Log_Price'] = np.log1p(df['Price (Per Annum)'])

In [6]:
df.head()

,District,Bedrooms,Price (Per Annum),Property Category,Property Type,Location,Scraped_At,Source,District_Tier,Log_Price
0,Kubwa,1,600000.0,Apartment,Self Contain,"Studio Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji,Tier 3 - Affordable,13.304687
1,Kubwa,1,1500000.0,Apartment,Apartment,"1bdrm Block of Flats in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji,Tier 3 - Affordable,14.220976
2,Life Camp,1,1200000.0,Apartment,Self Contain,"Studio Apartment in Efab City Estate, Life Cam...",2026-05-30 13:25:47,Jiji,Tier 2 - Mid-Market,13.997833
3,Kubwa,1,1200000.0,Apartment,Apartment,"1bdrm Apartment in Arab Road, Kubwa for rent",2026-05-30 13:25:47,Jiji,Tier 3 - Affordable,13.997833
4,Life Camp,2,1140000.0,Apartment,Apartment,Furnished 2bdrm Apartment in Life Camp for rent,2026-05-30 13:25:47,Jiji,Tier 2 - Mid-Market,13.946540


### one-hot encoding

In [7]:
# Create the dummy variables
categorical_cols_to_encode = ['Property Category', 'District_Tier', 'District']

df_encoded = pd.get_dummies(df, columns=categorical_cols_to_encode, drop_first=True, dtype=int)

print(df_encoded.shape)

(4300, 38)


### drop unnecessary columns

In [8]:
# Columns to drop
cols_to_drop = ['Location', 'Property Type', 'Scraped_At', 'Source']

df_final = df_encoded.drop(columns=cols_to_drop, errors='ignore')

df_final.head()

,Bedrooms,Price (Per Annum),Log_Price,Property Category_Bungalow,Property Category_Duplex,Property Category_House,District_Tier_Tier 2 - Mid-Market,District_Tier_Tier 3 - Affordable,District_Asokoro,District_Durumi,...,District_Kaura,District_Kubwa,District_Life Camp,District_Lokogoma,District_Lugbe,District_Mabushi,District_Maitama,District_Utako,District_Wuse,District_Wuye
0,1,600000.0,13.304687,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
1,1,1500000.0,14.220976,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
2,1,1200000.0,13.997833,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,1,1200000.0,13.997833,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
4,2,1140000.0,13.946540,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [9]:
df_final.shape

(4300, 34)

In [10]:
df_final.to_csv(MASTER / "processed_abuja_rentals.csv", index=False)